# Native barcoding for PromethION (SQK-NBD114.96)

Ligation sequencing of amplicons with the Native Barcoding Kit 96 V14, on the Hamilton STAR.
Protocol NBA_9170_v114_revS_15Jul2026. PromethION priming and loading stay manual.

The DNA plate is already normalized: **11.5 µL and 200 fmol per used well** (130 ng if the amplicon is 1 kb). Wells are filled column-wise from the first well, same as the nickel method. The barcode plate is the kit plate, columns oriented, so well A1 receives NB01.

You supply the end-prep master mix already combined (1.75 µL Ultra II buffer + 0.75 µL enzyme per sample, plus dead volume). The robot aliquots 2.5 µL. It does not pipette the enzyme and buffer separately.

**Tips.** Transfers under 10 µL use 10 µL tips. Pooling, beads, washes, and the 10 µL ligation buffer use 300 µL tips. DNA and barcodes are an 8-channel column transfer. Reagents that come from 1.5 mL tubes are single-channel, because those tubes are too narrow for eight channels. Each mix uses a fresh tip.

**Heat.** End-prep (20°C for 5 min, then 65°C for 5 min) and elutions (37°C for 10 min, agitate about every 2 min) are pauses on your thermocycler with a heated lid. The robot does not move the plate during a pause. Put it back on the same site. Room-temperature holds are on-deck timers. There is no vortexer, Hula mixer, or centrifuge: mixing is pipetting, and spin-downs are pauses.

**After EDTA** the robot pools every well into the 1.5 mL tube (12 µL each). Pool plus 0.4X beads must stay within 1400 µL, about 83 samples. The tip carrier limits a run from A1 to about 54 samples.

- Cleanup unchecked: the run waits. You do the protocol tube cleanup (0.4X AMPure, two 700 µL Short Fragment Buffer washes, elute in 35 µL water) and place 30 µL in cleanup-plate well A12.
- Cleanup checked: the robot does the 0.4X cleanup on the Alpaqua. A 700 µL wash does not fit in a PCR well, so the binding mixture is split to at most 150 µL per well and each well is washed twice with 120 µL Short Fragment Buffer. The eluate is brought to 35 µL before adapter ligation.

Adapter ligation is one PCR well: 30 µL sample, 5 µL Native Adapter, 10 µL Quick Ligation buffer, 5 µL Quick T4 ligase. The Alpaqua cleanup uses the protocol volumes (20 µL beads, two 125 µL washes, 25 µL Elution Buffer). Choose Short Fragment Buffer or Long Fragment Buffer for that wash. Ethanol is not used.

The magnet resource is a stand-in for the Alpaqua 96S Super Magnet. Replace `alpaqua_96s` when the real definition is ready, and teach `plate_z_offset`.

**Workflow.** Set the sample count, first well, and rails. Click Setup Deck. Restart the kernel before Setup Deck if you change those. The cleanup checkbox, DNA Control Sample, and adapter wash are read when you click Run. Click Continue at each pause.

| Carrier | Default rail | What is on it, front to back |
| --- | --- | --- |
| PCR plates, `PLT_CAR_L5PCR` | 1 | DNA / end-prep, barcodes, barcode ligation |
| Magnet MFX | 7 | Cleanup plate on the tapped holder, Alpaqua on the flat holder |
| Tips, `TIP_CAR_480_A00` | 13 | 10 µL racks, then one 300 µL rack |
| 1.5 mL tubes | 19 | Master mix, water, DCS, ligase, EDTA, Native Adapter, Quick Ligation buffer, Quick T4, AMPure, SFB, LFB, EB, elution water, pool tube |
| Waste trough | 21 | Liquid waste |


In [ ]:
import asyncio
import io
import math
import sys
import time
import traceback
from contextlib import redirect_stdout

import ipywidgets as widgets
from IPython.display import display

sys.path.insert(0, r"C:\Users\16122\Documents\Github\pylabrobot")

import pylabrobot.resources as resources
from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources import (
    Coordinate,
    PLT_CAR_L5PCR,
    TIP_CAR_480_A00,
    Trough_CAR_5R60_A00,
    alpaqua_96_plateadapter_magnum_flx,
    hamilton_1_trough_60ml_Vb,
    hamilton_96_tiprack_10uL,
    hamilton_96_tiprack_300uL,
    hamilton_mfx_carrier_L5_base,
    hamilton_mfx_plateholder_DWP_flat,
    hamilton_mfx_plateholder_DWP_metal_tapped,
    set_volume_tracking,
)
from pylabrobot.resources.hamilton import STARDeck

deck_is_ready = False
lh = None
status_output = None
continue_button = None
pause_event = asyncio.Event()
volumes = {}
cleanup_is_on_magnet = False
built_key = None

ten_column_tips = []
ten_single_tips = []
three_single_tips = []
ten_column_i = 0
ten_single_i = 0
three_single_i = 0


In [ ]:
# Protocol volumes, in uL. NBA_9170_v114_revS_15Jul2026.
DNA_UL = 11.5
DCS_OR_WATER_UL = 1.0
END_PREP_MIX_UL = 2.5
LIGATION_WATER_UL = 3.0
LIGATION_DNA_UL = 0.75
BARCODE_UL = 1.25
LIGASE_UL = 5.0
EDTA_UL = 2.0
POOLED_PER_SAMPLE_UL = 12.0
BEAD_RATIO = 0.4
TUBE_MAX_UL = 1400.0
WELL_MAX_UL = 150.0
PLATE_WASH_UL = 120.0
TUBE_WASH_UL = 700.0
ELUTE_TARGET_UL = 35.0
CONCENTRATE_ABOVE_UL = 40.0
MIN_ELUTE_PER_WELL_UL = 12.0
CONCENTRATE_BEAD_RATIO = 1.8
CONCENTRATE_PORTION_UL = 53.0
ADAPTER_SAMPLE_UL = 30.0
NA_UL = 5.0
LIGATION_BUFFER_UL = 10.0
T4_UL = 5.0
ADAPTER_BEADS_UL = 20.0
ADAPTER_WASH_UL = 125.0
ADAPTER_ELUTE_UL = 25.0
ADAPTER_WELL = "A12"
STAGING_WELLS = ["H12", "G12", "F12", "E12", "D12", "C12", "B12"]

MIX_CYCLES = 10
BIND_S = 10 * 60
LIGATION_S = 20 * 60
PELLET_S = 5 * 60
ADAPTER_PELLET_S = 2 * 60

SMALL_FLOW = 5.0
MEDIUM_FLOW = 15.0
LARGE_FLOW = 40.0
BEAD_FLOW = 20.0
VISCOUS_FLOW = 10.0

# Teach this: magnet hole bottom to the well cavity bottom, in mm.
# 0.62 is the cookbook starting point for a different plate on a Magnum FLX.
PLATE_Z_OFFSET = 0.62
CORE_GRIP_STRENGTH = 40

number_of_samples = 8
first_well = "A1"
use_dcs = False
cleanup_after_edta = True
adapter_wash = "SFB"

pcr_carrier_rail = 1
magnet_carrier_rail = 7
tip_carrier_rail = 13
tube_carrier_rail = 19
waste_trough_rail = 21

DEVELOPMENT_MODE = False

TUBE_ORDER = [
    "end_prep_master_mix",
    "water",
    "dcs",
    "blunt_ta_ligase",
    "edta",
    "native_adapter",
    "quick_ligation_buffer",
    "quick_t4_ligase",
    "ampure",
    "sfb",
    "lfb",
    "elution_buffer",
    "elution_water",
    "pool_tube",
]


In [ ]:
def log(message: str) -> None:
    text = message if message.endswith("\n") else message + "\n"
    if status_output is not None:
        status_output.append_stdout(text)
    else:
        print(text, end="")


def item(resource, name):
    found = resource[name]
    if isinstance(found, (list, tuple)):
        return found[0]
    return found


def plate_wells(plate_name: str, names: list[str]):
    plate = lh.deck.get_resource(plate_name)
    return [item(plate, name) for name in names]


def tube(name: str):
    return lh.deck.get_resource(name)


def sample_names(n: int, start: str) -> list[str]:
    """Column-wise 96-well names. Later columns start again at row A."""
    rows = list("ABCDEFGH")
    start = start.strip().upper()
    if len(start) < 2 or start[0] not in rows:
        raise ValueError(f"First well must look like A1, got {start!r}.")
    row_i = rows.index(start[0])
    col = int(start[1:])
    if col < 1 or col > 12:
        raise ValueError(f"First well column must be 1-12, got {start!r}.")
    names = []
    for column in range(col, 13):
        first_row = row_i if column == col else 0
        for row in rows[first_row:]:
            names.append(f"{row}{column}")
            if len(names) == n:
                return names
    raise ValueError(f"Only {len(names)} wells are left from {start}.")


def column_groups(names: list[str]) -> list[list[str]]:
    """Split well names into same-column groups so the 8 channels share one X."""
    groups: list[list[str]] = []
    current: list[str] = []
    for name in names:
        if current and name[1:] != current[0][1:]:
            groups.append(current)
            current = []
        current.append(name)
    if current:
        groups.append(current)
    return groups


def row_index(name: str) -> int:
    return "ABCDEFGH".index(name[0])


def pool_and_beads_ul(n: int) -> tuple[float, float, float]:
    pool = POOLED_PER_SAMPLE_UL * n
    beads = BEAD_RATIO * pool
    return pool, beads, pool + beads


def assert_pool_fits(n: int) -> None:
    pool, beads, total = pool_and_beads_ul(n)
    if total > TUBE_MAX_UL:
        raise ValueError(
            f"{n} samples make {total:.0f} uL of pool plus 0.4X beads. "
            f"The 1.5 mL tube limit is {TUBE_MAX_UL:.0f} uL."
        )


def binding_well_count(n: int) -> int:
    mixture = pool_and_beads_ul(n)[2]
    return max(1, math.ceil(mixture / WELL_MAX_UL - 1e-9))


def elution_volume_per_well(n_wells: int) -> float:
    if n_wells == 1:
        return ELUTE_TARGET_UL
    share = ELUTE_TARGET_UL / n_wells
    if share >= MIN_ELUTE_PER_WELL_UL:
        return share
    return 20.0


def portions(total: float, n: int) -> list[float]:
    if n <= 1:
        return [round(total, 2)]
    even = math.floor((total / n) * 100) / 100
    amounts = [even] * (n - 1)
    amounts.append(round(total - even * (n - 1), 2))
    return amounts


def tip_budget(n: int, start: str) -> dict:
    """10 uL tips: two 8-channel columns per sample column, plus 5 single tips per well and 2 adapter tips."""
    names = sample_names(n, start)
    groups = column_groups(names)
    eight_cols = 2 * len(groups)
    single_10 = 5 * n + 2
    spots_10 = eight_cols * 8 + single_10
    racks_10 = math.ceil(spots_10 / 96)
    if racks_10 + 1 > 5:
        raise ValueError(
            f"{n} samples starting at {start} need {racks_10} racks of 10 uL tips "
            f"({spots_10} tips) plus one 300 uL rack. The tip carrier has 5 positions."
        )
    return {
        "names": names,
        "groups": groups,
        "eight_cols": eight_cols,
        "single_10": single_10,
        "racks_10": racks_10,
        "binding_wells": binding_well_count(n),
    }


def load_report(n: int, start: str, dcs: bool, cleanup: bool, wash: str) -> list[str]:
    budget = tip_budget(n, start)
    pool, beads, mixture = pool_and_beads_ul(n)
    n_bind = budget["binding_wells"]
    each = elution_volume_per_well(n_bind)
    eluted = n_bind * each
    concentrate = cleanup and eluted > CONCENTRATE_ABOVE_UL
    n_stage = math.ceil(eluted / CONCENTRATE_PORTION_UL - 1e-9) if concentrate else 0
    water = (0.0 if dcs else DCS_OR_WATER_UL) * n + LIGATION_WATER_UL * n
    if cleanup:
        water += eluted + (ELUTE_TARGET_UL if concentrate else 0.0)
    water += 30
    ampure = ADAPTER_BEADS_UL + 30
    if cleanup:
        ampure += beads
        if concentrate:
            ampure += CONCENTRATE_BEAD_RATIO * eluted
    sfb = 40.0
    lfb = 40.0
    if cleanup:
        sfb += PLATE_WASH_UL * 2 * n_bind
        if concentrate:
            sfb += PLATE_WASH_UL * 2 * n_stage
    if wash == "SFB":
        sfb += ADAPTER_WASH_UL * 2
    else:
        lfb += ADAPTER_WASH_UL * 2
    lines = [
        f"Samples: {n} from {start} ({len(budget['groups'])} columns).",
        f"10 uL tip racks: {budget['racks_10']}. 300 uL tip racks: 1.",
        f"Pool {pool:.1f} uL. 0.4X beads {beads:.1f} uL. Mixture {mixture:.1f} uL.",
        "Load at least:",
        f" - End-prep master mix: {END_PREP_MIX_UL * n + 20:.0f} uL",
        f" - Water: {water:.0f} uL",
        f" - Diluted DCS: {(DCS_OR_WATER_UL * n + 15) if dcs else 0:.0f} uL",
        f" - Blunt/TA ligase master mix: {LIGASE_UL * n + 20:.0f} uL",
        f" - EDTA: {EDTA_UL * n + 15:.0f} uL",
        f" - Native Adapter: {NA_UL + 10:.0f} uL",
        f" - Quick Ligation buffer: {LIGATION_BUFFER_UL + 15:.0f} uL",
        f" - Quick T4 ligase: {T4_UL + 10:.0f} uL",
        f" - AMPure: {ampure:.0f} uL",
        f" - Short Fragment Buffer: {sfb:.0f} uL",
        f" - Long Fragment Buffer: {lfb:.0f} uL",
        f" - Elution Buffer: {ADAPTER_ELUTE_UL + 15:.0f} uL",
        f" - Elution water: {(eluted + (ELUTE_TARGET_UL if concentrate else 0) + 20) if cleanup else 0:.0f} uL",
    ]
    if cleanup:
        lines.append(
            f"Plate cleanup uses {n_bind} well(s), about {mixture / n_bind:.0f} uL of bead mixture each, "
            f"then two {PLATE_WASH_UL:.0f} uL SFB washes."
        )
        if concentrate:
            lines.append(
                f"Eluate is about {eluted:.0f} uL, so it is concentrated to {ELUTE_TARGET_UL:.0f} uL."
            )
    return lines


def self_check() -> None:
    assert sample_names(8, "A1") == [f"{row}1" for row in "ABCDEFGH"]
    assert sample_names(10, "A1")[8] == "A2"
    assert sample_names(3, "C1") == ["C1", "D1", "E1"]
    assert binding_well_count(8) == 1
    assert elution_volume_per_well(1) == 35.0
    assert binding_well_count(9) == 2
    assert abs(elution_volume_per_well(2) - 17.5) < 1e-9
    assert binding_well_count(24) == 3
    assert elution_volume_per_well(3) == 20.0
    try:
        assert_pool_fits(84)
    except ValueError:
        pass
    else:
        raise AssertionError("84 samples should not fit in the 1.5 mL tube.")
    assert tip_budget(48, "A1")["racks_10"] == 4
    try:
        tip_budget(55, "A1")
    except ValueError:
        pass
    else:
        raise AssertionError("55 samples should not fit on the tip carrier.")
    assert abs(sum(portions(60, 2)) - 60) < 1e-6
    assert_pool_fits(8)
    tip_budget(8, "A1")


self_check()


In [ ]:
def pcr_plate(name: str):
    """Eppendorf twin.tec 96 LoBind semi-skirted is the protocol plate (EP0030129504)."""
    maker = getattr(resources, "Eppendorf_96_wellplate_250ul_Vb_semiskirted", None)
    if maker is None:
        maker = resources.Eppendorf_96_wellplate_250ul_Vb
    return maker(name)


def lidded_tube(name: str):
    maker = getattr(resources, "eppendorf_tube_1500uL_Vb", None)
    if maker is None:
        maker = resources.Eppendorf_DNA_LoBind_1_5ml_Vb
    return maker(name)


def tube_carrier(name: str):
    maker = getattr(resources, "hamilton_tube_carrier_32_a00_insert_eppendorf_1_5mL", None)
    if maker is None:
        maker = resources.Tube_CAR_32_A00
    return maker(name)


def alpaqua_96s(name: str):
    """Stand-in for the Alpaqua 96S Super Magnet.

    PyLabRobot currently defines the Magnum FLX plate adapter, not the 96S.
    Replace this function with the 96S geometry when you add that resource.
    `plate_z_offset` is the distance from the magnet hole bottom to the well cavity bottom.
    """
    magnet = alpaqua_96_plateadapter_magnum_flx(name=name)
    magnet.plate_z_offset = PLATE_Z_OFFSET
    return magnet


def make_pcr_carrier(name: str):
    try:
        return PLT_CAR_L5PCR(name, with_adapters=True)
    except TypeError:
        return PLT_CAR_L5PCR(name)


def rack_columns(rack):
    columns = []
    for col in range(1, 13):
        column = []
        for row_i, row in enumerate("ABCDEFGH"):
            column.append((item(rack, f"{row}{col}"), row_i))
        columns.append(column)
    return columns


def reset_tip_cursors(racks_10: int) -> None:
    global ten_column_tips, ten_single_tips, three_single_tips
    global ten_column_i, ten_single_i, three_single_i
    columns = []
    for index in range(racks_10):
        columns.extend(rack_columns(lh.deck.get_resource(f"tips_10uL_{index + 1}")))
    budget = tip_budget(number_of_samples, first_well)
    reserved = columns[: budget["eight_cols"]]
    rest = columns[budget["eight_cols"] :]
    ten_column_tips = reserved
    ten_single_tips = [pair for column in rest for pair in column]
    three_single_tips = [pair for column in rack_columns(lh.deck.get_resource("tips_300uL")) for pair in column]
    ten_column_i = 0
    ten_single_i = 0
    three_single_i = 0


def take_column_tips(names: list[str]):
    global ten_column_i
    if ten_column_i >= len(ten_column_tips):
        raise RuntimeError("Ran out of 10 uL column tips.")
    column = ten_column_tips[ten_column_i]
    ten_column_i += 1
    return [column[row_index(name)] for name in names]


def take_single_tip(kind: str):
    global ten_single_i, three_single_i
    if kind == "10":
        if ten_single_i >= len(ten_single_tips):
            raise RuntimeError("Ran out of single 10 uL tips.")
        pair = ten_single_tips[ten_single_i]
        ten_single_i += 1
        return pair
    if three_single_i >= len(three_single_tips):
        raise RuntimeError("Ran out of 300 uL tips.")
    pair = three_single_tips[three_single_i]
    three_single_i += 1
    return pair


def vol_of(well) -> float:
    return volumes.get(id(well), 0.0)


def set_vol(well, amount: float) -> None:
    volumes[id(well)] = amount


def add_vol(well, amount: float) -> None:
    volumes[id(well)] = vol_of(well) + amount


def unique_rails(rails: dict[str, int]) -> None:
    used = list(rails.values())
    if len(set(used)) != len(used):
        raise ValueError(f"Carrier rails overlap: {rails}.")


def build_deck_resources():
    budget = tip_budget(number_of_samples, first_well)
    tip_car = TIP_CAR_480_A00(name="tip carrier")
    for index in range(budget["racks_10"]):
        tip_car[index] = hamilton_96_tiprack_10uL(name=f"tips_10uL_{index + 1}")
    tip_car[budget["racks_10"]] = hamilton_96_tiprack_300uL(name="tips_300uL")

    plates = make_pcr_carrier("pcr carrier")
    plates[0] = pcr_plate("dna_plate")
    plates[1] = pcr_plate("barcode_plate")
    plates[2] = pcr_plate("ligation_plate")

    cleanup_holder = hamilton_mfx_plateholder_DWP_metal_tapped(name="cleanup_holder")
    magnet_holder = hamilton_mfx_plateholder_DWP_flat(name="magnet_holder")
    magnet_holder.assign_child_resource(alpaqua_96s("alpaqua_96s"))
    magnet_carrier = hamilton_mfx_carrier_L5_base(
        name="magnet_carrier",
        modules={0: cleanup_holder, 1: magnet_holder},
    )
    magnet_carrier[0] = pcr_plate("cleanup_plate")

    tubes = tube_carrier("reagent tubes")
    for index, name in enumerate(TUBE_ORDER):
        tubes[index] = lidded_tube(name)

    waste_car = Trough_CAR_5R60_A00(name="waste carrier")
    waste_car[0] = hamilton_1_trough_60ml_Vb(name="liquid_waste")

    return {
        "tip_carrier": tip_car,
        "plates": plates,
        "magnet_carrier": magnet_carrier,
        "tubes": tubes,
        "waste_carrier": waste_car,
        "racks_10": budget["racks_10"],
    }


async def setup_deck():
    global lh, deck_is_ready, built_key
    if deck_is_ready or lh is not None:
        return "Deck setup already started. Restart the kernel to set it up again."

    try:
        assert_pool_fits(number_of_samples)
        tip_budget(number_of_samples, first_well)
        rails = {
            "PCR plates": pcr_carrier_rail,
            "Magnet": magnet_carrier_rail,
            "Tips": tip_carrier_rail,
            "Tubes": tube_carrier_rail,
            "Waste": waste_trough_rail,
        }
        unique_rails(rails)

        lh = LiquidHandler(
            backend=STARBackend(read_timeout=2400),
            deck=STARDeck(core_grippers="1000uL-5mL-on-waste"),
        )
        await lh.setup(skip_iswap=True)
        set_volume_tracking(enabled=False)

        resources_for_run = build_deck_resources()
        lh.deck.assign_child_resource(resources_for_run["plates"], rails=pcr_carrier_rail)
        lh.deck.assign_child_resource(resources_for_run["magnet_carrier"], rails=magnet_carrier_rail)
        lh.deck.assign_child_resource(resources_for_run["tip_carrier"], rails=tip_carrier_rail)
        lh.deck.assign_child_resource(resources_for_run["tubes"], rails=tube_carrier_rail)
        lh.deck.assign_child_resource(resources_for_run["waste_carrier"], rails=waste_trough_rail)
        lh.deck.get_resource("alpaqua_96s").plate_z_offset = PLATE_Z_OFFSET
        reset_tip_cursors(resources_for_run["racks_10"])

        built_key = (number_of_samples, first_well.strip().upper(), tuple(sorted(rails.items())))
        deck_is_ready = True
        buf = io.StringIO()
        with redirect_stdout(buf):
            lh.summary()
        note = (
            "Cleanup plate is on the tapped holder. The Alpaqua stand-in is on the flat holder. "
            f"plate_z_offset is {PLATE_Z_OFFSET:.2f} mm. "
            "Tube 1 is the front of the tube carrier.\n\n"
            + "\n".join(load_report(number_of_samples, first_well, use_dcs, cleanup_after_edta, adapter_wash))
        )
        return {"status": "Deck setup complete.\n\n" + note, "deck": buf.getvalue().strip()}
    except Exception:
        partial = lh
        lh = None
        deck_is_ready = False
        built_key = None
        if partial is not None and getattr(partial, "stop", None) is not None:
            try:
                await partial.stop()
            except Exception:
                pass
        raise


In [ ]:
async def operator_pause(message: str) -> None:
    log(message)
    log("Click Continue when the labware is back on the same site.")
    pause_event.clear()
    continue_button.disabled = False
    await pause_event.wait()
    continue_button.disabled = True


async def wait_seconds(seconds: float, label: str) -> None:
    log(f"{label}: waiting {seconds / 60:.0f} min.")
    await asyncio.sleep(seconds)
    log(f"{label}: done.")


async def transfer_one(kind: str, source, dest, volume: float, flow: float) -> None:
    remaining = volume
    tip_max = 9.5 if kind == "10" else 280.0
    while remaining > 0.05:
        chunk = min(remaining, tip_max)
        spot, channel = take_single_tip(kind)
        await lh.pick_up_tips([spot], use_channels=[channel])
        await lh.aspirate(
            [source],
            vols=[chunk],
            flow_rates=[flow],
            use_channels=[channel],
            offsets=[Coordinate(0, 0, 1)],
        )
        await lh.dispense(
            [dest],
            vols=[chunk],
            flow_rates=[flow],
            use_channels=[channel],
            offsets=[Coordinate(0, 0, 2)],
        )
        await lh.discard_tips(use_channels=[channel])
        remaining -= chunk


async def mix_one(kind: str, well, volume: float, flow: float, cycles: int) -> None:
    if cycles <= 0 or volume < 0.5:
        return
    spot, channel = take_single_tip(kind)
    await lh.pick_up_tips([spot], use_channels=[channel])
    for _ in range(cycles):
        await lh.aspirate(
            [well],
            vols=[volume],
            flow_rates=[flow],
            use_channels=[channel],
            offsets=[Coordinate(0, 0, 1)],
        )
        await lh.dispense(
            [well],
            vols=[volume],
            flow_rates=[flow],
            use_channels=[channel],
            offsets=[Coordinate(0, 0, 3)],
        )
    await lh.discard_tips(use_channels=[channel])


async def add_and_mix(source, dest, volume: float, flow: float, cycles: int) -> None:
    """Transfer and mix with one tip. 10 uL tips mix wells up to 20 uL; larger wells use 300 uL tips."""
    kind = "10" if volume < 10 else "300"
    tip_max = 9.5 if kind == "10" else 280.0
    spot, channel = take_single_tip(kind)
    await lh.pick_up_tips([spot], use_channels=[channel])
    remaining = volume
    while remaining > 0.05:
        chunk = min(remaining, tip_max)
        await lh.aspirate(
            [source],
            vols=[chunk],
            flow_rates=[flow],
            use_channels=[channel],
            offsets=[Coordinate(0, 0, 1)],
        )
        await lh.dispense(
            [dest],
            vols=[chunk],
            flow_rates=[flow],
            use_channels=[channel],
            offsets=[Coordinate(0, 0, 2)],
        )
        remaining -= chunk
    add_vol(dest, volume)
    present = vol_of(dest)
    if kind == "10" and present > 20:
        await lh.discard_tips(use_channels=[channel])
        await mix_one("300", dest, min(200.0, present * 0.75), flow, cycles)
        return
    mix_cap = 8.0 if kind == "10" else 200.0
    mix_vol = min(mix_cap, present * 0.75)
    for _ in range(cycles):
        if mix_vol < 0.5:
            break
        await lh.aspirate(
            [dest],
            vols=[mix_vol],
            flow_rates=[flow],
            use_channels=[channel],
            offsets=[Coordinate(0, 0, 1)],
        )
        await lh.dispense(
            [dest],
            vols=[mix_vol],
            flow_rates=[flow],
            use_channels=[channel],
            offsets=[Coordinate(0, 0, 3)],
        )
    await lh.discard_tips(use_channels=[channel])


async def transfer_column(src_plate: str, dst_plate: str, names: list[str], volume: float, flow: float, cycles: int) -> None:
    pairs = take_column_tips(names)
    spots = [pair[0] for pair in pairs]
    channels = [pair[1] for pair in pairs]
    src = plate_wells(src_plate, names)
    dst = plate_wells(dst_plate, names)
    n = len(names)
    await lh.pick_up_tips(spots, use_channels=channels)
    await lh.aspirate(
        src,
        vols=[volume] * n,
        flow_rates=[flow] * n,
        use_channels=channels,
        offsets=[Coordinate(0, 0, 1)] * n,
    )
    await lh.dispense(
        dst,
        vols=[volume] * n,
        flow_rates=[flow] * n,
        use_channels=channels,
        offsets=[Coordinate(0, 0, 2)] * n,
    )
    for well in dst:
        add_vol(well, volume)
    mix_vols = [min(8.0, max(vol_of(well) * 0.75, 0.5)) for well in dst]
    for _ in range(cycles):
        await lh.aspirate(
            dst,
            vols=mix_vols,
            flow_rates=[flow] * n,
            use_channels=channels,
            offsets=[Coordinate(0, 0, 1)] * n,
        )
        await lh.dispense(
            dst,
            vols=mix_vols,
            flow_rates=[flow] * n,
            use_channels=channels,
            offsets=[Coordinate(0, 0, 3)] * n,
        )
    await lh.discard_tips(use_channels=channels)


async def move_cleanup(on_magnet: bool) -> None:
    global cleanup_is_on_magnet
    if on_magnet == cleanup_is_on_magnet:
        return
    plate = lh.deck.get_resource("cleanup_plate")
    target_name = "alpaqua_96s" if on_magnet else "cleanup_holder"
    target = lh.deck.get_resource(target_name)
    log("Moving the cleanup plate onto the Alpaqua." if on_magnet else "Moving the cleanup plate home.")
    await lh.move_plate(
        plate,
        to=target,
        pickup_distance_from_top=6,
        use_arm="core",
        core_grip_strength=CORE_GRIP_STRENGTH,
        return_core_gripper=False,
    )
    check = getattr(lh.backend, "core_check_resource_exists_at_location_center", None)
    if check is not None:
        await check(
            location=plate.get_absolute_location(),
            resource=plate,
            gripper_y_margin=9,
            enable_recovery=True,
            audio_feedback=False,
        )
    cleanup_is_on_magnet = on_magnet
    if not on_magnet:
        await lh.backend.return_core_gripper_tools()


async def remove_to_waste(plate_name: str, names: list[str]) -> None:
    waste = tube("liquid_waste")
    for name in names:
        well = item(lh.deck.get_resource(plate_name), name)
        remaining = vol_of(well)
        while remaining > 0.05:
            chunk = min(remaining, 280.0)
            spot, channel = take_single_tip("300")
            await lh.pick_up_tips([spot], use_channels=[channel])
            await lh.aspirate(
                [well],
                vols=[chunk],
                flow_rates=[BEAD_FLOW],
                use_channels=[channel],
                offsets=[Coordinate(0, 0, 0.5)],
            )
            await lh.dispense(
                [waste],
                vols=[chunk],
                flow_rates=[LARGE_FLOW],
                use_channels=[channel],
            )
            await lh.discard_tips(use_channels=[channel])
            remaining -= chunk
        set_vol(well, 0.0)


async def wash_plate_wells(names: list[str], buffer_name: str, wash_ul: float, repeats: int) -> None:
    buffer = tube(buffer_name)
    for repeat in range(repeats):
        log(f"Wash {repeat + 1} of {repeats}: {wash_ul:.0f} uL {buffer_name.upper()} per well.")
        await move_cleanup(False)
        for name in names:
            well = item(lh.deck.get_resource("cleanup_plate"), name)
            await add_and_mix(buffer, well, wash_ul, LARGE_FLOW, 8)
        await operator_pause(
            "Spin the cleanup plate so the wash sits in the wells, then return it to the home site."
        )
        await move_cleanup(True)
        await wait_seconds(PELLET_S, "Pelleting wash")
        await remove_to_waste("cleanup_plate", names)


In [ ]:
async def end_prep(names: list[str]) -> None:
    log("End-prep: 1 uL DCS or water, then 2.5 uL master mix.")
    source_name = "dcs" if use_dcs else "water"
    for name in names:
        well = item(lh.deck.get_resource("dna_plate"), name)
        set_vol(well, DNA_UL)
        await add_and_mix(tube(source_name), well, DCS_OR_WATER_UL, SMALL_FLOW, MIX_CYCLES)
    for name in names:
        well = item(lh.deck.get_resource("dna_plate"), name)
        await add_and_mix(tube("end_prep_master_mix"), well, END_PREP_MIX_UL, SMALL_FLOW, MIX_CYCLES)
    await operator_pause(
        "Seal the DNA plate. Spin it down. On the thermocycler, with a heated lid, "
        "incubate at 20°C for 5 min, then 65°C for 5 min. Spin again, unseal, and "
        "return the plate to the same site."
    )


async def barcode_ligation(names: list[str], groups: list[list[str]]) -> None:
    log("Barcode ligation in the new plate. Mix between water, DNA, barcode, and ligase.")
    for name in names:
        well = item(lh.deck.get_resource("ligation_plate"), name)
        set_vol(well, 0.0)
        await add_and_mix(tube("water"), well, LIGATION_WATER_UL, SMALL_FLOW, MIX_CYCLES)
    for group in groups:
        log(f"Transfer {LIGATION_DNA_UL} uL end-prepped DNA for {group[0]}:{group[-1]}.")
        await transfer_column("dna_plate", "ligation_plate", group, LIGATION_DNA_UL, SMALL_FLOW, MIX_CYCLES)
    for group in groups:
        log(f"Transfer {BARCODE_UL} uL barcode for {group[0]}:{group[-1]}.")
        await transfer_column("barcode_plate", "ligation_plate", group, BARCODE_UL, SMALL_FLOW, MIX_CYCLES)
    for name in names:
        well = item(lh.deck.get_resource("ligation_plate"), name)
        await add_and_mix(tube("blunt_ta_ligase"), well, LIGASE_UL, VISCOUS_FLOW, MIX_CYCLES)
    await operator_pause("Spin the ligation plate and return it to the same site.")
    await wait_seconds(LIGATION_S, "Barcode ligation at room temperature")


async def add_edta_and_pool(names: list[str]) -> None:
    log("Adding 2 uL EDTA to each ligation well.")
    for name in names:
        well = item(lh.deck.get_resource("ligation_plate"), name)
        await add_and_mix(tube("edta"), well, EDTA_UL, SMALL_FLOW, MIX_CYCLES)
    await operator_pause("Spin the ligation plate and return it to the same site.")
    pool = tube("pool_tube")
    set_vol(pool, 0.0)
    spot, channel = take_single_tip("300")
    await lh.pick_up_tips([spot], use_channels=[channel])
    for name in names:
        well = item(lh.deck.get_resource("ligation_plate"), name)
        await lh.aspirate(
            [well],
            vols=[POOLED_PER_SAMPLE_UL],
            flow_rates=[MEDIUM_FLOW],
            use_channels=[channel],
            offsets=[Coordinate(0, 0, 1)],
        )
        await lh.dispense(
            [pool],
            vols=[POOLED_PER_SAMPLE_UL],
            flow_rates=[MEDIUM_FLOW],
            use_channels=[channel],
            offsets=[Coordinate(0, 0, 2)],
        )
        add_vol(pool, POOLED_PER_SAMPLE_UL)
    await lh.discard_tips(use_channels=[channel])
    log(f"Pooled {vol_of(pool):.1f} uL into the 1.5 mL tube.")


async def manual_cleanup_pause(n: int) -> None:
    pool, beads, _total = pool_and_beads_ul(n)
    await operator_pause(
        "The pool is in the 1.5 mL tube. Finish this cleanup by hand: "
        f"add {beads:.1f} uL AMPure (0.4X of {pool:.1f} uL), mix, and bind 10 min. "
        f"Pellet, and wash twice with {TUBE_WASH_UL:.0f} uL Short Fragment Buffer. "
        f"Elute in {ELUTE_TARGET_UL:.0f} uL nuclease-free water at 37°C for 10 min, "
        "agitating about every 2 min. "
        f"Place {ADAPTER_SAMPLE_UL:.0f} uL of that eluate into cleanup-plate well {ADAPTER_WELL}. "
        "Leave the cleanup plate on its home site."
    )
    set_vol(item(lh.deck.get_resource("cleanup_plate"), ADAPTER_WELL), ADAPTER_SAMPLE_UL)


async def robot_barcode_cleanup(n: int) -> str:
    pool_well = tube("pool_tube")
    _pool, beads, mixture = pool_and_beads_ul(n)
    log(f"Adding {beads:.1f} uL AMPure to the pool.")
    await add_and_mix(tube("ampure"), pool_well, beads, BEAD_FLOW, 8)
    n_bind = binding_well_count(n)
    bind_names = sample_names(n_bind, "A1")
    if ADAPTER_WELL in bind_names or any(name[1:] == "12" for name in bind_names):
        raise RuntimeError("Binding wells overlap the adapter well. Use fewer samples.")
    amounts = portions(mixture, n_bind)
    log(f"Splitting the bead mixture across {', '.join(bind_names)}.")
    for name, amount in zip(bind_names, amounts):
        dest = item(lh.deck.get_resource("cleanup_plate"), name)
        set_vol(dest, 0.0)
        await add_and_mix(pool_well, dest, amount, BEAD_FLOW, 6)
    await wait_seconds(BIND_S, "Bead binding")
    await operator_pause("Spin the cleanup plate and return it to the home site.")
    await move_cleanup(True)
    await wait_seconds(PELLET_S, "Pelleting the barcode cleanup")
    await remove_to_waste("cleanup_plate", bind_names)
    await wash_plate_wells(bind_names, "sfb", PLATE_WASH_UL, 2)
    each = elution_volume_per_well(n_bind)
    log(f"Eluting each binding well in {each:.1f} uL water.")
    await move_cleanup(False)
    for name in bind_names:
        dest = item(lh.deck.get_resource("cleanup_plate"), name)
        await add_and_mix(tube("elution_water"), dest, each, MEDIUM_FLOW, 8)
    await operator_pause(
        "Take the cleanup plate from its home site. Incubate at 37°C for 10 min "
        "and agitate the elution wells about every 2 min. Spin, and return the plate home."
    )
    await move_cleanup(True)
    await wait_seconds(ADAPTER_PELLET_S, "Pelleting the elution")
    eluted = each * n_bind
    if eluted <= CONCENTRATE_ABOVE_UL:
        dest_name = bind_names[0] if n_bind == 1 else "H12"
        if n_bind > 1:
            dest = item(lh.deck.get_resource("cleanup_plate"), dest_name)
            set_vol(dest, 0.0)
            for name in bind_names:
                src = item(lh.deck.get_resource("cleanup_plate"), name)
                await transfer_one("300", src, dest, vol_of(src), MEDIUM_FLOW)
                add_vol(dest, vol_of(src))
                set_vol(src, 0.0)
        return dest_name
    return await concentrate_eluate(bind_names)


async def concentrate_eluate(bind_names: list[str]) -> str:
    total = sum(vol_of(item(lh.deck.get_resource("cleanup_plate"), name)) for name in bind_names)
    n_stage = math.ceil(total / CONCENTRATE_PORTION_UL - 1e-9)
    if n_stage > len(STAGING_WELLS):
        raise RuntimeError(f"Eluate of {total:.0f} uL needs {n_stage} staging wells.")
    stage_names = STAGING_WELLS[:n_stage]
    amounts = portions(total, n_stage)
    log(f"Concentrating {total:.1f} uL of eluate in {', '.join(stage_names)}.")
    sources = list(bind_names)
    staged = []
    for stage_name, amount in zip(stage_names, amounts):
        dest = item(lh.deck.get_resource("cleanup_plate"), stage_name)
        set_vol(dest, 0.0)
        still = amount
        for source_name in sources:
            if still <= 0.05:
                break
            src = item(lh.deck.get_resource("cleanup_plate"), source_name)
            take = min(vol_of(src), still)
            if take <= 0.05:
                continue
            await transfer_one("300", src, dest, take, MEDIUM_FLOW)
            add_vol(dest, take)
            set_vol(src, vol_of(src) - take)
            still -= take
        staged.append((dest, CONCENTRATE_BEAD_RATIO * vol_of(dest)))
    await move_cleanup(False)
    for dest, bead_volume in staged:
        await add_and_mix(tube("ampure"), dest, bead_volume, BEAD_FLOW, 8)
    await wait_seconds(BIND_S, "Concentration bind")
    await operator_pause("Spin the cleanup plate and return it to the home site.")
    await move_cleanup(True)
    await wait_seconds(PELLET_S, "Pelleting the concentration")
    await remove_to_waste("cleanup_plate", stage_names)
    await wash_plate_wells(stage_names, "sfb", PLATE_WASH_UL, 2)
    await move_cleanup(False)
    first = item(lh.deck.get_resource("cleanup_plate"), stage_names[0])
    await add_and_mix(tube("elution_water"), first, ELUTE_TARGET_UL, MEDIUM_FLOW, 8)
    await operator_pause(
        "Incubate the cleanup plate at 37°C for 10 min and agitate the elution well "
        "about every 2 min. Spin, and return the plate home."
    )
    current_name = stage_names[0]
    for next_name in stage_names[1:]:
        await move_cleanup(True)
        await wait_seconds(ADAPTER_PELLET_S, "Pelleting serial elution")
        src = item(lh.deck.get_resource("cleanup_plate"), current_name)
        dest = item(lh.deck.get_resource("cleanup_plate"), next_name)
        take = vol_of(src)
        await transfer_one("300", src, dest, take, MEDIUM_FLOW)
        set_vol(src, 0.0)
        set_vol(dest, take)
        await move_cleanup(False)
        await mix_one("300", dest, min(200.0, take * 0.75), MEDIUM_FLOW, 8)
        await operator_pause(
            f"Incubate the cleanup plate at 37°C for 10 min so well {next_name} can elute. "
            "Agitate about every 2 min. Spin, and return the plate home."
        )
        current_name = next_name
    await move_cleanup(True)
    await wait_seconds(ADAPTER_PELLET_S, "Pelleting the final elution")
    return current_name


async def collect_adapter_sample(eluate_well: str) -> None:
    if eluate_well == ADAPTER_WELL:
        return
    await move_cleanup(True)
    src = item(lh.deck.get_resource("cleanup_plate"), eluate_well)
    dest = item(lh.deck.get_resource("cleanup_plate"), ADAPTER_WELL)
    if vol_of(src) < ADAPTER_SAMPLE_UL:
        raise RuntimeError(
            f"Well {eluate_well} has {vol_of(src):.1f} uL. Adapter ligation needs {ADAPTER_SAMPLE_UL:.0f} uL."
        )
    set_vol(dest, 0.0)
    await transfer_one("300", src, dest, ADAPTER_SAMPLE_UL, MEDIUM_FLOW)
    add_vol(dest, ADAPTER_SAMPLE_UL)
    set_vol(src, vol_of(src) - ADAPTER_SAMPLE_UL)
    log(f"Moved {ADAPTER_SAMPLE_UL:.0f} uL into well {ADAPTER_WELL}.")


async def adapter_ligation() -> None:
    dest = item(lh.deck.get_resource("cleanup_plate"), ADAPTER_WELL)
    await move_cleanup(False)
    log(f"Adapter ligation in well {ADAPTER_WELL}.")
    await add_and_mix(tube("native_adapter"), dest, NA_UL, SMALL_FLOW, MIX_CYCLES)
    await add_and_mix(tube("quick_ligation_buffer"), dest, LIGATION_BUFFER_UL, MEDIUM_FLOW, MIX_CYCLES)
    await add_and_mix(tube("quick_t4_ligase"), dest, T4_UL, VISCOUS_FLOW, MIX_CYCLES)
    await operator_pause("Spin the cleanup plate and return it to the home site.")
    await wait_seconds(LIGATION_S, "Adapter ligation at room temperature")
    await add_and_mix(tube("ampure"), dest, ADAPTER_BEADS_UL, BEAD_FLOW, 8)
    await wait_seconds(BIND_S, "Adapter bead binding")
    await operator_pause("Spin the cleanup plate and return it to the home site.")
    await move_cleanup(True)
    await wait_seconds(ADAPTER_PELLET_S, "Pelleting the adapter cleanup")
    await remove_to_waste("cleanup_plate", [ADAPTER_WELL])
    buffer_name = "sfb" if adapter_wash == "SFB" else "lfb"
    await wash_plate_wells([ADAPTER_WELL], buffer_name, ADAPTER_WASH_UL, 2)
    await move_cleanup(False)
    await add_and_mix(tube("elution_buffer"), dest, ADAPTER_ELUTE_UL, MEDIUM_FLOW, 8)
    await operator_pause(
        "Incubate the cleanup plate at 37°C for 10 min and agitate well A12 about every 2 min. "
        "Spin, and return the plate home."
    )
    await move_cleanup(True)
    await wait_seconds(ADAPTER_PELLET_S, "Pelleting the library elution")
    await lh.backend.return_core_gripper_tools()


def promethion_message() -> str:
    return (
        f"The cleanup plate is on the Alpaqua. The library supernatant is in well {ADAPTER_WELL}, "
        f"in {ADAPTER_ELUTE_UL:.0f} uL Elution Buffer. Leave the plate on the magnet while you take the supernatant. "
        "Quantify 1 uL on the Qubit. PromethION load targets: "
        "100 fmol if the library is under 1 kb, 35-50 fmol if it is 1-10 kb, "
        "and 300 ng if it is over 10 kb. Prime and load the flow cell by hand."
    )


async def run_barcoding() -> None:
    global volumes, cleanup_is_on_magnet
    if not deck_is_ready or lh is None or built_key is None:
        raise RuntimeError("Deck is not ready. Click Setup Deck first.")
    rails = {
        "PCR plates": pcr_carrier_rail,
        "Magnet": magnet_carrier_rail,
        "Tips": tip_carrier_rail,
        "Tubes": tube_carrier_rail,
        "Waste": waste_trough_rail,
    }
    key = (number_of_samples, first_well.strip().upper(), tuple(sorted(rails.items())))
    if key != built_key:
        raise RuntimeError("Sample count, first well, or rails changed. Restart the kernel and click Setup Deck.")

    volumes = {}
    cleanup_is_on_magnet = False
    names = sample_names(number_of_samples, first_well)
    groups = column_groups(names)
    for line in load_report(number_of_samples, first_well, use_dcs, cleanup_after_edta, adapter_wash):
        log(line)

    await end_prep(names)
    await barcode_ligation(names, groups)
    await add_edta_and_pool(names)
    if cleanup_after_edta:
        eluate_well = await robot_barcode_cleanup(number_of_samples)
        await collect_adapter_sample(eluate_well)
        await move_cleanup(False)
        await operator_pause(
            f"Optional Qubit: well {eluate_well} still has leftover eluate. "
            f"Well {ADAPTER_WELL} already holds the {ADAPTER_SAMPLE_UL:.0f} uL for adapter ligation. "
            "Return the plate to the home site."
        )
    else:
        await manual_cleanup_pause(number_of_samples)
    await adapter_ligation()
    log(promethion_message())
    await operator_pause(promethion_message())


In [ ]:
STYLE = {"description_width": "180px"}


def flt(value, low, high, step, description):
    return widgets.BoundedFloatText(
        value=value, min=low, max=high, step=step, description=description, style=STYLE
    )


def intbox(value, low, high, description):
    return widgets.BoundedIntText(value=value, min=low, max=high, description=description, style=STYLE)


def apply_ui_settings() -> None:
    global number_of_samples, first_well, use_dcs, cleanup_after_edta, adapter_wash
    global MIX_CYCLES, PLATE_Z_OFFSET, DEVELOPMENT_MODE
    global pcr_carrier_rail, magnet_carrier_rail, tip_carrier_rail, tube_carrier_rail, waste_trough_rail
    number_of_samples = int(samples_field.value)
    first_well = first_well_field.value.strip().upper()
    use_dcs = bool(dcs_toggle.value)
    cleanup_after_edta = bool(cleanup_toggle.value)
    adapter_wash = wash_field.value
    MIX_CYCLES = int(mix_field.value)
    PLATE_Z_OFFSET = float(offset_field.value)
    DEVELOPMENT_MODE = bool(development_toggle.value)
    pcr_carrier_rail = int(pcr_rail_field.value)
    magnet_carrier_rail = int(magnet_rail_field.value)
    tip_carrier_rail = int(tip_rail_field.value)
    tube_carrier_rail = int(tube_rail_field.value)
    waste_trough_rail = int(waste_rail_field.value)


def format_duration(seconds: float) -> str:
    total = max(0, int(round(seconds)))
    hours, rem = divmod(total, 3600)
    minutes, secs = divmod(rem, 60)
    if hours and minutes:
        return f"{hours} h {minutes} min"
    if hours:
        return f"{hours} h"
    if minutes:
        return f"{minutes} min"
    return f"{secs} s"


def estimate_run_seconds(n: int, groups: int, cleanup: bool, mixes: int) -> float:
    """Robot time for transfers, mixes, and on-deck waits. Thermocycler pauses are extra."""
    single = (5 * n + 8) * (16 + mixes * 4)
    columns = (2 * groups) * (20 + mixes * 5)
    waits = 2 * LIGATION_S + BIND_S + ADAPTER_PELLET_S
    if cleanup:
        waits += BIND_S + PELLET_S
    return single + columns + waits


title = widgets.HTML("<h3>Native barcoding, SQK-NBD114.96</h3>")
instructions = widgets.HTML(
    "<b>Workflow:</b> set the sample count, first well, and rails, then click <b>Setup Deck</b>. "
    "Restart the kernel before Setup Deck if you change those. "
    "DNA Control, the EDTA cleanup, and the adapter wash are read when you click <b>Run barcoding</b>. "
    "The DNA plate already holds 11.5 uL per sample. Click <b>Continue</b> at each pause."
)
samples_field = intbox(number_of_samples, 1, 83, "Samples")
first_well_field = widgets.Text(value=first_well, description="First well", style=STYLE)
dcs_toggle = widgets.Checkbox(value=use_dcs, description="Add 1 uL diluted DCS")
cleanup_toggle = widgets.Checkbox(
    value=cleanup_after_edta,
    description="Cleanup after EDTA on the Alpaqua",
    layout={"width": "360px"},
)
wash_field = widgets.Dropdown(
    options=["SFB", "LFB"],
    value=adapter_wash,
    description="Adapter wash",
    style=STYLE,
)
mix_field = intbox(MIX_CYCLES, 1, 20, "Mix cycles")
offset_field = flt(PLATE_Z_OFFSET, -5, 20, 0.01, "Plate Z offset mm")
development_toggle = widgets.Checkbox(value=False, description="Development mode (tracebacks)")
pcr_rail_field = intbox(pcr_carrier_rail, -4, 60, "PCR plate rail")
magnet_rail_field = intbox(magnet_carrier_rail, -4, 60, "Magnet rail")
tip_rail_field = intbox(tip_carrier_rail, -4, 60, "Tip rail")
tube_rail_field = intbox(tube_carrier_rail, -4, 60, "Tube rail")
waste_rail_field = intbox(waste_trough_rail, -4, 60, "Waste rail")

setup_button = widgets.Button(description="Setup Deck", button_style="info")
run_button = widgets.Button(description="Run barcoding", button_style="success", disabled=True)
continue_button = widgets.Button(description="Continue", button_style="warning", disabled=True)
return_8_button = widgets.Button(description="Return 8-head tips", button_style="warning", disabled=True)
trash_8_button = widgets.Button(description="Trash 8-head tips", button_style="danger", disabled=True)
status_output = widgets.Output(
    layout={"border": "1px solid #ddd", "padding": "8px", "max_height": "360px", "overflow": "auto"}
)
deck_summary = widgets.Textarea(
    value="",
    description="Deck",
    layout={"width": "100%", "height": "220px"},
    disabled=True,
)
run_state_label = widgets.HTML("<b>Status:</b> Idle")
eta_label = widgets.HTML("")


def _set_busy(busy: bool) -> None:
    setup_button.disabled = busy or deck_is_ready
    ready = deck_is_ready and not busy
    run_button.disabled = not ready
    return_8_button.disabled = not ready
    trash_8_button.disabled = not ready
    if busy:
        continue_button.disabled = True


def _show_error(exc: Exception) -> None:
    status_output.append_stdout(f"Failed: {exc}\n")
    if DEVELOPMENT_MODE:
        status_output.append_stdout("\n" + "".join(traceback.format_exc()) + "\n")


def _refresh_eta(_change=None) -> None:
    try:
        apply_ui_settings()
        budget = tip_budget(number_of_samples, first_well)
        assert_pool_fits(number_of_samples)
        seconds = estimate_run_seconds(
            number_of_samples,
            len(budget["groups"]),
            cleanup_after_edta,
            MIX_CYCLES,
        )
        eta_label.value = (
            "<div style='font-size:18px;padding:10px 14px;margin:4px 0 8px;"
            "background:#eef5ff;border:1px solid #b7d0ef;border-radius:6px;'>"
            f"<b>Estimated robot time: {format_duration(seconds)}</b>"
            "<div style='font-size:13px;color:#333;margin-top:4px;'>"
            "Thermocycler pauses and spin pauses are not included."
            "</div></div>"
        )
    except Exception as exc:
        eta_label.value = (
            "<div style='font-size:18px;padding:10px 14px;margin:4px 0 8px;"
            "background:#fff6e8;border:1px solid #e6c98a;border-radius:6px;'>"
            f"<b>Check the sample settings.</b> {exc}"
            "</div>"
        )


async def _setup_async():
    _set_busy(True)
    if not DEVELOPMENT_MODE:
        status_output.clear_output()
    status_output.append_stdout("Setting up deck...\n")
    try:
        apply_ui_settings()
        result = await setup_deck()
        if isinstance(result, dict):
            status_output.append_stdout(result["status"] + "\n")
            deck_summary.value = result["deck"] or "Deck is ready."
        else:
            status_output.append_stdout(str(result) + "\n")
    except Exception as exc:
        _show_error(exc)
    finally:
        _set_busy(False)


async def _run_async():
    _set_busy(True)
    if not DEVELOPMENT_MODE:
        status_output.clear_output()
    status_output.append_stdout("Running native barcoding...\n")
    status_output.append_stdout(time.ctime() + "\n")
    try:
        apply_ui_settings()
        budget = tip_budget(number_of_samples, first_well)
        seconds = estimate_run_seconds(
            number_of_samples, len(budget["groups"]), cleanup_after_edta, MIX_CYCLES
        )
        finish = time.strftime("%I:%M %p", time.localtime(time.time() + seconds)).lstrip("0")
        run_state_label.value = (
            f"<b>Status:</b> Running. Estimated robot finish {finish} ({format_duration(seconds)}), "
            "plus thermocycler pauses."
        )
        await run_barcoding()
        status_output.append_stdout("Run complete.\n")
        status_output.append_stdout(time.ctime() + "\n")
    except Exception as exc:
        _show_error(exc)
    finally:
        run_state_label.value = "<b>Status:</b> Idle"
        _set_busy(False)


async def _return_8_async():
    _set_busy(True)
    try:
        if not deck_is_ready or lh is None:
            raise RuntimeError("Deck is not ready. Click Setup Deck first.")
        await lh.return_tips(allow_nonzero_volume=True)
        status_output.append_stdout("Returned 8-head tips.\n")
    except Exception as exc:
        _show_error(exc)
    finally:
        _set_busy(False)


async def _trash_8_async():
    _set_busy(True)
    try:
        if not deck_is_ready or lh is None:
            raise RuntimeError("Deck is not ready. Click Setup Deck first.")
        await lh.discard_tips(allow_nonzero_volume=True)
        status_output.append_stdout("Trashed 8-head tips.\n")
    except Exception as exc:
        _show_error(exc)
    finally:
        _set_busy(False)


def _on_continue(_button) -> None:
    pause_event.set()


for widget in (
    samples_field,
    first_well_field,
    dcs_toggle,
    cleanup_toggle,
    wash_field,
    mix_field,
):
    widget.observe(_refresh_eta, names="value")
_refresh_eta()

setup_button.on_click(lambda _: asyncio.create_task(_setup_async()))
run_button.on_click(lambda _: asyncio.create_task(_run_async()))
continue_button.on_click(_on_continue)
return_8_button.on_click(lambda _: asyncio.create_task(_return_8_async()))
trash_8_button.on_click(lambda _: asyncio.create_task(_trash_8_async()))

advanced = widgets.Accordion(
    children=[
        widgets.VBox(
            [
                mix_field,
                offset_field,
                widgets.HTML("Rails. PCR and magnet carriers are 6 tracks wide. Tubes and the waste trough are 1 track."),
                pcr_rail_field,
                magnet_rail_field,
                tip_rail_field,
                tube_rail_field,
                waste_rail_field,
                development_toggle,
            ]
        )
    ]
)
advanced.set_title(0, "Rails, mix, and magnet offset")

display(
    widgets.VBox(
        [
            title,
            instructions,
            eta_label,
            samples_field,
            first_well_field,
            dcs_toggle,
            cleanup_toggle,
            wash_field,
            advanced,
            widgets.HBox([setup_button, run_button, continue_button, return_8_button, trash_8_button]),
            run_state_label,
            status_output,
            deck_summary,
        ]
    )
)
